In [1]:
import pandas as pd 
import numpy as np 
from statsmodels.stats.proportion import proportions_ztest

In [2]:
df = pd.read_csv("online_shoppers_intention.csv")

In [3]:
df.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [6]:
#a/b testing for Weekend
result = df.groupby('Weekend').agg(
        visitors = ('Revenue', 'count'),
        purchases = ("Revenue", 'sum')
).reset_index()
result['conversion_rate'] = result['purchases']/result['visitors']*100
print(result)

   Weekend  visitors  purchases  conversion_rate
0    False      9462       1409        14.891144
1     True      2868        499        17.398884


In [11]:
visits = result['visitors']
purchase = result['purchases']
z_test, p_value = proportions_ztest(purchase, visits)
print("The z_test value is: ", z_test)
print("The p-value is : ", p_value)
if p_value<0.05:
    print("The Alternate Hypothesis is Statistically Correct")
else:
    print("The Null Hypothesis is Statistically Correct")

The z_test value is:  -3.2529732781407104
The p-value is :  0.0011420423637110938
The Alternate Hypothesis is Statistically Correct


In [17]:
df['VisitorType'].value_counts()

VisitorType
Returning_Visitor    10551
New_Visitor           1694
Other                   85
Name: count, dtype: int64

In [18]:
#Since the 'Other' category in vistortype is ver small we will drop it 
df_ab = df[df['VisitorType'] != 'Other']

In [19]:
df_ab['VisitorType'].value_counts()

VisitorType
Returning_Visitor    10551
New_Visitor           1694
Name: count, dtype: int64

In [21]:
# a/b testing for VistorType : 'NewVisitor' vs 'Returning Visitor'
result = df_ab.groupby("VisitorType").agg(
        visitors = ("Revenue", 'count'),
        purchases = ("Revenue", "sum")
).reset_index()
result['conversion_rate'] = result['purchases']/result['visitors']*100
print(result)

         VisitorType  visitors  purchases  conversion_rate
0        New_Visitor      1694        422        24.911452
1  Returning_Visitor     10551       1470        13.932329


In [22]:
visits = result['visitors']
purchase = result['purchases']
z_test, p_value = proportions_ztest(purchase, visits)
print("The z_test value is: ", z_test)
print("The p-value is : ", p_value)
if p_value<0.05:
    print("The Alternate Hypothesis is Statistically Correct")
else:
    print("The Null Hypothesis is Statistically Correct")

The z_test value is:  11.605314290218352
The p-value is :  3.8726217006968784e-31
The Alternate Hypothesis is Statistically Correct


In [23]:
#a/b testing for ProductRelated pages
#segmenting the productRelated pages into HIGH and LOW 
# We are using median because it well help to segment the data into two group based on the median value found 
product_related = df['ProductRelated'].median()
df['product_group'] = np.where(
          df['ProductRelated'] >= product_related,
            'High',
             'Low'
)

In [26]:
result =  df.groupby('product_group').agg(
    visitors = ('Revenue', 'count'),
    purchases = ('Revenue', 'sum')
).reset_index()
result['conversion_rate'] =  result['purchases']/result["visitors"]*100
print(result)

  product_group  visitors  purchases  conversion_rate
0          High      6240       1339        21.458333
1           Low      6090        569         9.343186


In [27]:
visits = result['visitors']
purchase = result['purchases']
z_test, p_value = proportions_ztest(purchase, visits)
print("The z_test value is: ", z_test)
print("The p-value is : ", p_value)
if p_value<0.05:
    print("The Alternate Hypothesis is Statistically Correct")
else:
    print("The Null Hypothesis is Statistically Correct")

The z_test value is:  18.597153510726894
The p-value is :  3.3882790612577407e-77
The Alternate Hypothesis is Statistically Correct


In [30]:
#a/b testing for Product Related Duration
# Creating 'High' and 'Low' categories for Product Related Duration
median_duration = df["ProductRelated_Duration"].median()
df['duration_groups'] = np.where(
        df['ProductRelated_Duration'] >= median_duration,
        'High',
        'Low'
) 
result =  df.groupby('duration_groups').agg(
    visitors = ('Revenue', 'count'),
    purchases = ('Revenue', 'sum')
).reset_index()
result['conversion_rate'] =  result['purchases']/result["visitors"]*100
print(result)

  duration_groups  visitors  purchases  conversion_rate
0            High      6165       1374        22.287105
1             Low      6165        534         8.661800


In [31]:
visits = result['visitors']
purchase = result['purchases']
z_test, p_value = proportions_ztest(purchase, visits)
print("The z_test value is: ", z_test)
print("The p-value is : ", p_value)
if p_value<0.05:
    print("The Alternate Hypothesis is Statistically Correct")
else:
    print("The Null Hypothesis is Statistically Correct")

The z_test value is:  20.916841464509773
The p-value is :  3.7622112071127655e-97
The Alternate Hypothesis is Statistically Correct


In [35]:
# a/b testing for bounce rates
# creating 'High' and 'Low' categories for Bounce Rates
median_bounce =  df['BounceRates'].median()
df['bounce_groups'] = np.where(
            df['BounceRates'] >= median_bounce,
            'High',
            'Low'
)
result =  df.groupby('bounce_groups').agg(
    visitors = ('Revenue', 'count'),
    purchases = ('Revenue', 'sum')
).reset_index()
result['conversion_rate'] =  result['purchases']/result["visitors"]*100
print(result)

  bounce_groups  visitors  purchases  conversion_rate
0          High      6165        704        11.419303
1           Low      6165       1204        19.529603


In [36]:
visits = result['visitors']
purchase = result['purchases']
z_test, p_value = proportions_ztest(purchase, visits)
print("The z_test value is: ", z_test)
print("The p-value is : ", p_value)
if p_value<0.05:
    print("The Alternate Hypothesis is Statistically Correct")
else:
    print("The Null Hypothesis is Statistically Correct")

The z_test value is:  -12.450500871732007
The p-value is :  1.3897028573513229e-35
The Alternate Hypothesis is Statistically Correct
